# Solutions to Select Homework Exercises

In [1]:
# This is a code cell that imports the necessary libraries for our session.
import numpy as np                        # NumPy for numerical computations
import scipy as sp                        # SciPy for scientific computing
import sympy as sym                       # SymPy for symbolic mathematics
import matplotlib as mpl                  # Matplotlib for plotting
import matplotlib.pyplot as plt           # Matplotlib pyplot interface
from scipy.integrate import solve_ivp     # ODE solver
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

This page collects full solutions to select homework exercises. Each
solution is worked out “by hand” and then, where helpful, confirmed
symbolically using SymPy or other computational tools.

## Homework 1

### Section 1.1 Exercise 9, page 10

Find two values of $\lambda$ for which $x(t) = e^{\lambda t}$ is a
solution of the differential equation $2x'' - 5x' - 3x = 0$.

#### By Hand

**Step 1 — Differentiate the candidate solution.**

For $x(t) = e^{\lambda t}$, $$
x'(t) = \lambda e^{\lambda t}, \qquad x''(t) = \lambda^2 e^{\lambda t}.
$$

**Step 2 — Substitute into the differential equation.**

$$
2x'' - 5x' - 3x = 2\lambda^2 e^{\lambda t} - 5\lambda e^{\lambda t} - 3 e^{\lambda t} = \left(2\lambda^2 - 5\lambda - 3\right)e^{\lambda t}.
$$

Since $e^{\lambda t} \neq 0$ for any $t$, this expression is zero
exactly when

$$
2\lambda^2 - 5\lambda - 3 = 0.
$$

This is the **characteristic equation** associated with the ODE.

**Step 3 — Solve the characteristic equation.**

Factoring, $$
2\lambda^2 - 5\lambda - 3 = (2\lambda + 1)(\lambda - 3) = 0,
$$ so $$
\lambda = -\frac{1}{2} \qquad \text{or} \qquad \lambda = 3.
$$

> **Tip**
>
> Both values work independently: $x_1(t) = e^{-t/2}$ and
> $x_2(t) = e^{3t}$ are each solutions of $2x'' - 5x' - 3x = 0$, and so
> is any linear combination $x(t) = c_1 e^{-t/2} + c_2 e^{3t}$.

#### Using SymPy

In [2]:
t, lam = sym.symbols('t lambda', real=True)

x = sym.exp(lam * t)

lhs = 2 * sym.diff(x, t, 2) - 5 * sym.diff(x, t) - 3 * x
char_eq = sym.simplify(lhs / x)   # divide out the common factor e^{lambda t}

print("2x'' - 5x' - 3x  simplifies to:", sym.expand(char_eq), " * e^(lambda t)")

lambda_solutions = sym.solve(sym.Eq(char_eq, 0), lam)
print("Values of lambda:", lambda_solutions)

2x'' - 5x' - 3x  simplifies to: 2*lambda**2 - 5*lambda - 3  * e^(lambda t)
Values of lambda: [-1/2, 3]

> **Note**
>
> SymPy confirms the two characteristic roots found by hand:
> $\lambda = -\dfrac{1}{2}$ and $\lambda = 3$.

------------------------------------------------------------------------

### Section 1.1 Exercise 12, page 11

*(Physics)* In deep water, the intensity of light $I = I(x)$ at a depth
$x$ meters below the water surface is modeled by the equation
$I' = -1.4I$. At what depth is the light intensity $1\%$ that at the
surface?

#### By Hand

**Step 1 — Solve the differential equation.**

The equation $I' = -1.4I$ is of the form $I'=aI$ where $a = -1.4$, so we
know from lecture that the general solution must be $I(x) = I_0 e^{ax}$
for some constant $I_0$.

**Step 2 — Impose the $1\%$ condition.**

We want the depth $x$ at which $I(x) = 0.01\,I_0$: $$
0.01\,I_0 = I_0 e^{-1.4x} \quad \Longrightarrow \quad 0.01 = e^{-1.4x}.
$$

**Step 3 — Solve for $x$.**

Taking the natural log of both sides, $$
\ln(0.01) = -1.4x \quad \Longrightarrow \quad x = -\frac{\ln(0.01)}{1.4} = \frac{\ln(100)}{1.4}.
$$

Numerically, $$
x = \frac{\ln(100)}{1.4} \approx \frac{4.6052}{1.4} \approx 3.29 \text{ meters}.
$$

> **Tip**
>
> **Interpretation.** Since $1.4 > 0$ is the decay rate, light intensity
> decreases exponentially with depth. At roughly $3.29$ meters below the
> surface, only $1\%$ of the surface intensity remains — a useful
> benchmark for how quickly light is absorbed in deep water.

## Homework 2

### Section 1.1.3 — Slope Fields

#### Exercise 6, Page 6

Use software to sketch the slope field for the differential equation
$x' = x^2 - t$ on the square $-3 < t < 3,\ -3 < x < 3$.

**Setting up the slope field.**

A slope field is produced by evaluating the right-hand side
$f(t,x) = x^2 - t$ at a grid of points $(t,x)$ in the given square, and
at each point drawing a short line segment with that slope. Since no
closed-form solution is needed, the natural way to “solve” this exercise
is with code.

In [3]:
def f(t, x):
    return x**2 - t

# Grid of points for the slope field arrows
t_grid = np.linspace(-3, 3, 21)
x_grid = np.linspace(-3, 3, 21)
T, X = np.meshgrid(t_grid, x_grid)
slopes = f(T, X)

# Normalize direction vectors so every arrow has the same visual length
dt = np.ones_like(slopes)
dx = slopes
norm = np.sqrt(dt**2 + dx**2)
dt_unit, dx_unit = dt / norm, dx / norm

fig, ax = plt.subplots(figsize=(6, 6))
ax.quiver(T, X, dt_unit, dx_unit, angles='xy', pivot='mid',
          headwidth=0, headlength=0, headaxislength=0,
          color='steelblue', width=0.003)

# Overlay a handful of numerically computed solution curves
for x0 in [-2.5, -1, 0, 1, 2.5]:
    sol = solve_ivp(f, [0, 3], [x0], dense_output=True, max_step=0.05)
    ax.plot(sol.t, sol.y[0], color='darkorange', lw=1.5)
    sol_back = solve_ivp(f, [0, -3], [x0], dense_output=True, max_step=0.05)
    ax.plot(sol_back.t, sol_back.y[0], color='darkorange', lw=1.5)

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('$t$', fontsize=13)
ax.set_ylabel('$x$', fontsize=13)
ax.set_title(r"Slope field for $x' = x^2 - t$", fontsize=13)
plt.tight_layout()
plt.show()

> **Note**
>
> There is no elementary closed-form solution for $x' = x^2 - t$ (it is
> a form of **Riccati equation**), so this exercise is intentionally
> solved by producing the slope field and, optionally, a few
> numerically-generated solution curves rather than an explicit formula.

------------------------------------------------------------------------

### Section 1.3.1 — Separable Equations

#### Exercise 4(a), Page 26

Find the general solution: $x' = \dfrac{2x}{t+1}$.

**By Hand.**

Separate variables (assuming $x \neq 0$ and $t \neq -1$): $$
\frac{dx}{x} = \frac{2}{t+1}\,dt.
$$

Integrating both sides, $$
\ln|x| = 2\ln|t+1| + C.
$$

Exponentiating, $$
|x| = e^{C}(t+1)^2 \quad \Longrightarrow \quad x(t) = A(t+1)^2,
$$ where $A$ is an arbitrary constant (absorbing the sign and $e^C$).
Note $A = 0$ recovers the trivial solution $x \equiv 0$, so this formula
captures **all** solutions.

$$
\boxed{x(t) = A(t+1)^2}
$$

In [4]:
t = sym.symbols('t')
x = sym.Function('x')

eq = sym.Eq(sym.diff(x(t), t), 2*x(t)/(t+1))
sol = sym.dsolve(eq, x(t))
print("General solution:", sol)

General solution: Eq(x(t), C1*(t**2 + 2*t + 1))

> **Note**
>
> SymPy returns $x(t) = C_1(t+1)^2$, matching the boxed formula above
> (with $A = C_1$).

#### Exercise 10, Page 27

Solve the following initial value problems.

##### (a) $\dfrac{dx}{dt} = e^{t+x}, \quad x(0) = 0$.

**By Hand.**

Rewrite the right-hand side as $e^t e^x$ and separate variables: $$
e^{-x}\,dx = e^{t}\,dt.
$$

Integrating, $$
-e^{-x} = e^{t} + C.
$$

Apply $x(0) = 0$:
$-e^{0} = e^{0} + C \Rightarrow -1 = 1 + C \Rightarrow C = -2$. So $$
-e^{-x} = e^t - 2 \quad \Longrightarrow \quad e^{-x} = 2 - e^t \quad \Longrightarrow \quad x(t) = -\ln\!\left(2 - e^t\right).
$$

$$
\boxed{x(t) = -\ln(2 - e^t)}, \qquad t < \ln 2.
$$

> **Tip**
>
> The restriction $t < \ln 2$ is required for $2 - e^t > 0$, so that the
> logarithm is defined. This is an example of a solution whose
> **interval of existence** is limited even though the differential
> equation itself is defined for all $t$ and $x$.

In [5]:
xf = sym.Function('x')
eq10a = sym.Eq(sym.diff(xf(t), t), sym.exp(t + xf(t)))
sol10a = sym.dsolve(eq10a, xf(t), ics={xf(0): 0})
print("Solution:", sym.simplify(sol10a))

Solution: Eq(x(t), log(-1/(exp(t) - 2)))

##### (b) $\dfrac{dT}{dt} = 2at\left(T^2 - a^2\right), \quad T(0) = 0$.

**By Hand.**

Separate variables: $$
\frac{dT}{T^2 - a^2} = 2at\,dt.
$$

Using the partial fraction decomposition
$\dfrac{1}{T^2-a^2} = \dfrac{1}{2a}\left(\dfrac{1}{T-a} - \dfrac{1}{T+a}\right)$
and integrating both sides, $$
\frac{1}{2a}\ln\left|\frac{T-a}{T+a}\right| = at^2 + C.
$$

Applying $T(0) = 0$: the left side is
$\dfrac{1}{2a}\ln\left|\dfrac{-a}{a}\right| = \dfrac{1}{2a}\ln(1) = 0$,
so $C = 0$. Thus $$
\ln\left|\frac{T-a}{T+a}\right| = 2a^2t^2 \quad \Longrightarrow \quad \frac{T-a}{T+a} = \pm\, e^{2a^2t^2}.
$$

The initial condition $T(0)=0$ forces
$\dfrac{T-a}{T+a}\Big|_{t=0} = -1$, so the sign is negative: $$
\frac{T-a}{T+a} = -e^{2a^2t^2}.
$$

Solving for $T$: $$
T - a = -e^{2a^2t^2}(T+a) \quad \Longrightarrow \quad T\left(1+e^{2a^2t^2}\right) = a\left(1 - e^{2a^2t^2}\right)
$$ $$
\Longrightarrow \quad T = a\,\frac{1 - e^{2a^2t^2}}{1+e^{2a^2t^2}} = -a\,\frac{e^{2a^2t^2}-1}{e^{2a^2t^2}+1}.
$$

Using the identity $\tanh(u) = \dfrac{e^{2u}-1}{e^{2u}+1}$ with
$u = a^2t^2$, this simplifies to

$$
\boxed{T(t) = -a\tanh\!\left(a^2t^2\right)}.
$$

In [6]:
a, t = sym.symbols('a t', positive=True)
Tf = sym.Function('T')

# Verify the closed-form solution directly, since dsolve with these ics
# runs into the usual difficulty of picking out one branch of a
# multivalued implicit solution.
T_guess = -a * sym.tanh(a**2 * t**2)

lhs = sym.diff(T_guess, t)
rhs = 2*a*t*(T_guess**2 - a**2)
print("Residual dT/dt - 2at(T^2 - a^2):", sym.simplify(lhs - rhs))
print("T(0) =", T_guess.subs(t, 0))

Residual dT/dt - 2at(T^2 - a^2): 0
T(0) = 0

> **Note**
>
> SymPy confirms both that the proposed closed-form
> $T(t) = -a\tanh(a^2t^2)$ satisfies the differential equation exactly
> (the residual simplifies to $0$) and that it meets the initial
> condition $T(0) = 0$.

##### (c) $\dfrac{dy}{dt} = t^2\tan y, \quad y(0) = 0$.

**By Hand.**

Notice that $y \equiv 0$ makes both sides zero: the left side is
$\frac{d}{dt}(0) = 0$, and the right side is $t^2\tan(0) = 0$. Since
$y(0) = 0$ matches this constant function, and the right-hand side
$t^2\tan y$ is smooth (hence Lipschitz) near $y = 0$, the initial value
problem has a **unique** solution — and it must be the equilibrium
solution.

$$
\boxed{y(t) \equiv 0}
$$

> **Tip**
>
> It is instructive to see where a naive separation of variables leads:
> dividing by $\tan y$ gives $\cot y\,dy = t^2\,dt$, so
> $\ln|\sin y| = \tfrac{1}{3}t^3 + C$. This step implicitly assumes
> $\sin y \neq 0$, silently discarding the constant solution
> $y \equiv 0$ — exactly the solution the initial condition selects.
> This is a good reminder to always check for equilibrium (constant)
> solutions before dividing them away.

In [7]:
t = sym.symbols('t')
yf = sym.Function('y')
eq10c = sym.Eq(sym.diff(yf(t), t), t**2 * sym.tan(yf(t)))

# The zero function should satisfy the ODE identically
residual = sym.diff(0, t) - t**2 * sym.tan(0)
print("Residual for y = 0:", residual)

Residual for y = 0: 0

------------------------------------------------------------------------

### Section 1.4.1 — First-Order Linear Equations & Integrating Factors

#### Exercise 2(a), Page 42

Find the general solution: $x' = -\dfrac{2}{t}x + t$.

**By Hand.**

Write the equation in standard linear form: $$
x' + \frac{2}{t}x = t.
$$

The integrating factor is $$
\mu(t) = \exp\!\left(\int \frac{2}{t}\,dt\right) = \exp\!\left(2\ln|t|\right) = t^2.
$$

Multiplying both sides by $\mu(t) = t^2$, the left side becomes an exact
derivative: $$
\left(t^2 x\right)' = t^2 \cdot t = t^3.
$$

Integrating, $$
t^2 x = \frac{t^4}{4} + C.
$$

$$
\boxed{x(t) = \frac{t^2}{4} + \frac{C}{t^2}}
$$

In [8]:
t = sym.symbols('t')
x = sym.Function('x')

eq2a = sym.Eq(sym.diff(x(t), t), -(2/t)*x(t) + t)
sol2a = sym.dsolve(eq2a, x(t))
print("General solution:", sol2a)

General solution: Eq(x(t), (C1 + t**4/4)/t**2)

> **Note**
>
> SymPy’s result
> $x(t) = \dfrac{C_1 + t^4/4}{t^2} = \dfrac{C_1}{t^2} + \dfrac{t^2}{4}$
> agrees with the boxed formula.

#### Exercise 3(a), Page 42

Solve the initial value problem:
$x' + \dfrac{5}{t}x = 1+t, \quad x(1) = 1$.

**By Hand.**

The equation is already in standard linear form with $p(t) = 5/t$. The
integrating factor is $$
\mu(t) = \exp\!\left(\int \frac{5}{t}\,dt\right) = e^{5\ln|t|} = t^5.
$$

Multiplying through by $t^5$: $$
\left(t^5 x\right)' = t^5(1+t) = t^5 + t^6.
$$

Integrating, $$
t^5 x = \frac{t^6}{6} + \frac{t^7}{7} + C \quad \Longrightarrow \quad x(t) = \frac{t}{6} + \frac{t^2}{7} + \frac{C}{t^5}.
$$

Applying $x(1) = 1$: $$
1 = \frac{1}{6} + \frac{1}{7} + C \quad \Longrightarrow \quad C = 1 - \frac{13}{42} = \frac{29}{42}.
$$

$$
\boxed{x(t) = \frac{t}{6} + \frac{t^2}{7} + \frac{29}{42\,t^5}}
$$

In [9]:
t = sym.symbols('t')
x = sym.Function('x')

eq3a = sym.Eq(sym.diff(x(t), t) + (5/t)*x(t), 1 + t)
sol3a = sym.dsolve(eq3a, x(t), ics={x(1): 1})
print("Particular solution:", sym.simplify(sol3a))

# Compare against the boxed formula
manual = t/6 + t**2/7 + sym.Rational(29, 42)/t**5
residual = sym.simplify(manual - sol3a.rhs)
print("Difference from by-hand formula:", residual)

Particular solution: Eq(x(t), (t**6*(6*t + 7) + 29)/(42*t**5))
Difference from by-hand formula: 0

> **Note**
>
> SymPy’s particular solution agrees exactly with the by-hand formula
> (the difference simplifies to $0$).

------------------------------------------------------------------------

### Section 1.4.3 — Applications: RC Circuits

#### Exercise 1, Page 53

Write the equation that governs an RC circuit with a 12-volt battery,
taking $R = 1$ and $C = \tfrac{1}{2}$. Determine the equilibrium
solution and its stability. If $Q(0) = 5$, find a formula for $Q(t)$.
Find the current $I(t)$. Plot the charge and the current on the same set
of axes.

**By Hand.**

**Step 1 — Set up the governing equation.**

For an RC circuit with a constant voltage source $V$, resistance $R$,
and capacitance $C$, the charge $Q(t)$ on the capacitor satisfies $$
R\frac{dQ}{dt} + \frac{Q}{C} = V \quad \Longrightarrow \quad \frac{dQ}{dt} = \frac{V}{R} - \frac{Q}{RC}.
$$

Substituting $V = 12$, $R = 1$, $C = \tfrac12$: $$
\frac{dQ}{dt} = 12 - 2Q.
$$

**Step 2 — Find the equilibrium solution and its stability.**

Setting $\dfrac{dQ}{dt} = 0$ gives $12 - 2Q_{\text{eq}} = 0$, so
$Q_{\text{eq}} = 6$. Writing the equation as $$
\frac{dQ}{dt} = -2(Q - 6),
$$ we see that $Q > 6 \Rightarrow \dfrac{dQ}{dt} < 0$ and
$Q < 6 \Rightarrow \dfrac{dQ}{dt} > 0$: solutions are always pushed back
toward $Q = 6$. Hence $Q_{\text{eq}} = 6$ is a **stable** equilibrium.

**Step 3 — Solve for $Q(t)$ with $Q(0) = 5$.**

This is linear with integrating factor $\mu(t) = e^{2t}$: $$
\left(Qe^{2t}\right)' = 12e^{2t} \quad \Longrightarrow \quad Qe^{2t} = 6e^{2t} + C \quad \Longrightarrow \quad Q(t) = 6 + Ce^{-2t}.
$$

Applying $Q(0) = 5$: $5 = 6 + C \Rightarrow C = -1$.

$$
\boxed{Q(t) = 6 - e^{-2t}}
$$

**Step 4 — Find the current $I(t)$.**

The current is the rate of change of charge: $$
I(t) = \frac{dQ}{dt} = \frac{d}{dt}\left(6 - e^{-2t}\right).
$$

$$
\boxed{I(t) = 2e^{-2t}}
$$

In [10]:
t = sym.symbols('t')
Q = sym.Function('Q')

eqQ = sym.Eq(sym.diff(Q(t), t), 12 - 2*Q(t))
solQ = sym.dsolve(eqQ, Q(t), ics={Q(0): 5})
print("Q(t):", solQ)

I_expr = sym.diff(solQ.rhs, t)
print("I(t) = dQ/dt:", I_expr)

Q(t): Eq(Q(t), 6 - exp(-2*t))
I(t) = dQ/dt: 2*exp(-2*t)

> **Note**
>
> SymPy confirms $Q(t) = 6 - e^{-2t}$, and differentiating gives
> $I(t) = 2e^{-2t}$, matching both boxed results.

**Step 5 — Plot $Q(t)$ and $I(t)$.**

In [11]:
t_vals = np.linspace(0, 4, 400)
Q_vals = 6 - np.exp(-2*t_vals)
I_vals = 2*np.exp(-2*t_vals)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_vals, Q_vals, color='steelblue', lw=2, label='$Q(t)$ (charge)')
ax.plot(t_vals, I_vals, color='darkorange', lw=2, label='$I(t)$ (current)')
ax.axhline(6, color='gray', linestyle='--', lw=1, label='Equilibrium $Q_{eq} = 6$')

ax.set_xlabel('$t$', fontsize=13)
ax.set_ylabel('Charge / Current', fontsize=13)
ax.set_title('RC Circuit: Charge and Current vs. Time', fontsize=13)
ax.legend(fontsize=10, loc='center right')
plt.tight_layout()
plt.show()

## Homework 3

### Section 1.5.1 — Autonomous Systems

#### Exercise 2(b)

For the model $x' = x(4-x)(5-x)^2$: (a) plot the growth rate $f(x)$
versus $x$ and sketch the phase line diagram; (b) find the equilibria
analytically and classify them according to their stability using
Theorem 1.32; (c) draw a few key time series plots, $x = x(t)$, in the
$tx$-plane.

**(a) & (b) — Equilibria, stability, and the phase line.**

Setting $f(x) = x(4-x)(5-x)^2 = 0$ gives three equilibria:
$x = 0,\ 4,\ 5$.

Theorem 1.32 classifies a *hyperbolic* equilibrium (one where
$f'(x_{eq}) \neq 0$) by the sign of $f'(x_{eq})$: negative means stable,
positive means unstable. Differentiating and evaluating, $$
f'(0) = 100 > 0 \ (\text{unstable}), \qquad f'(4) = -4 < 0 \ (\text{stable}), \qquad f'(5) = 0 \ (\text{inconclusive}).
$$

Since $f'(5) = 0$, Theorem 1.32 does not directly apply at $x=5$ and we
examine the sign of $f(x)$ directly instead. Because $(5-x)^2 \geq 0$
never changes sign, the sign of $f(x)$ near $x=5$ is controlled by
$x(4-x)$, which is negative for $x$ near $5$ on **both** sides.
Tabulating the sign of $f$ on each interval:

| Interval       | $(-\infty, 0)$ | $(0,4)$ | $(4,5)$ | $(5,\infty)$ |
|----------------|----------------|---------|---------|--------------|
| sign of $f(x)$ | $-$            | $+$     | $-$     | $-$          |

- At $x=0$: sign goes $- \to +$, so trajectories are pushed away on both
  sides $\Rightarrow$ **unstable**.
- At $x=4$: sign goes $+ \to -$, so trajectories are pulled in from both
  sides $\Rightarrow$ **stable**.
- At $x=5$: the sign is negative on both sides, so trajectories from
  below move *away* from $5$ (down toward $4$) while trajectories from
  above move *toward* $5$ $\Rightarrow$ **semistable**.

In [12]:
def f2b(x):
    return x*(4-x)*(5-x)**2

xmin, xmax = -1, 6
x = np.linspace(xmin, xmax, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x, f2b(x), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(0, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)          # unstable
ax1.plot(4, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)             # stable
ax1.plot(5, 0, marker='o', ms=10, fillstyle='right', mfc='seagreen',
          markerfacecoloralt='crimson', mec='black', zorder=5)                          # semistable
ax1.set_ylim(-60, 60)
ax1.set_ylabel('$f(x)$')
ax1.set_title(r"Growth rate $f(x) = x(4-x)(5-x)^2$")

bounds = [xmin, 0, 4, 5, xmax]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(f2b(mid))
    x0, x1 = (a + 0.15, b - 0.15) if sign > 0 else (b - 0.15, a + 0.15)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(0, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax2.plot(4, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.plot(5, 0, marker='o', ms=10, fillstyle='right', mfc='seagreen',
          markerfacecoloralt='crimson', mec='black', zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$x$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

In [13]:
x = sym.symbols('x')
f = x*(4-x)*(5-x)**2

eqs = sym.solve(sym.Eq(f, 0), x)
fp = sym.diff(f, x)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", fp.subs(x, e))

Equilibria: [0, 4, 5]
  f'(0) = 100
  f'(4) = -4
  f'(5) = 0

> **Note**
>
> SymPy confirms the equilibria $x = 0, 4, 5$ and the derivative values
> used above; the vanishing derivative at $x=5$ is exactly what forced
> the direct sign analysis.

**(c) — Representative time series.**

In [14]:
def f2b_t(t, x):
    return x*(4-x)*(5-x)**2

fig, ax = plt.subplots(figsize=(7, 5))
for x0 in [-0.5, 0.5, 1, 3, 4.5, 5, 5.5, 6]:
    sol = solve_ivp(f2b_t, [0, 3], [x0], dense_output=True, max_step=0.01, rtol=1e-8)
    ax.plot(sol.t, sol.y[0], lw=2, label=f'$x_0={x0}$')

for level in [0, 4, 5]:
    ax.axhline(level, color='gray', linestyle='--', lw=1)

ax.set_ylim(-3, 7)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Representative solutions for $x' = x(4-x)(5-x)^2$")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 2(g)

For the model $x' = (4-x)(2-x)^3$: (a) plot the growth rate $f(x)$
versus $x$ and sketch the phase line diagram; (b) find the equilibria
analytically and classify them according to their stability using
Theorem 1.32; (c) draw a few key time series plots, $x = x(t)$, in the
$tx$-plane.

**(a) & (b) — Equilibria, stability, and the phase line.**

Setting $f(x) = (4-x)(2-x)^3 = 0$ gives two equilibria: $x = 2$ (a
triple root of the factor $(2-x)^3$) and $x = 4$.

$$
f'(2) = 0 \ (\text{inconclusive}), \qquad f'(4) = 8 > 0 \ (\text{unstable}).
$$

Since $f'(2)=0$ (the triple root makes $x=2$ non-hyperbolic), we again
check signs directly:

| Interval       | $(-\infty, 2)$ | $(2,4)$ | $(4,\infty)$ |
|----------------|----------------|---------|--------------|
| sign of $f(x)$ | $+$            | $-$     | $+$          |

- At $x=2$: sign goes $+ \to -$ $\Rightarrow$ **stable**.
- At $x=4$: sign goes $- \to +$, consistent with $f'(4)>0$ $\Rightarrow$
  **unstable**.

In [15]:
def f2g(x):
    return (4-x)*(2-x)**3

xmin, xmax = 0, 6
x = np.linspace(xmin, xmax, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(x, f2g(x), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)     # stable
ax1.plot(4, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5) # unstable
ax1.set_ylim(-30, 30)
ax1.set_ylabel('$f(x)$')
ax1.set_title(r"Growth rate $f(x) = (4-x)(2-x)^3$")

bounds = [xmin, 2, 4, xmax]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(f2g(mid))
    x0, x1 = (a + 0.15, b - 0.15) if sign > 0 else (b - 0.15, a + 0.15)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.plot(4, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$x$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

In [16]:
x = sym.symbols('x')
f = (4-x)*(2-x)**3

eqs = sym.solve(sym.Eq(f, 0), x)
fp = sym.diff(f, x)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", fp.subs(x, e))

Equilibria: [2, 4]
  f'(2) = 0
  f'(4) = 8

**(c) — Representative time series.**

In [17]:
def f2g_t(t, x):
    return (4-x)*(2-x)**3

fig, ax = plt.subplots(figsize=(7, 5))
for x0 in [0.5, 1.5, 2, 3, 3.9, 4, 4.1, 4.5]:
    sol = solve_ivp(f2g_t, [0, 2], [x0], dense_output=True, max_step=0.005, rtol=1e-9)
    ax.plot(sol.t, sol.y[0], lw=2, label=f'$x_0={x0}$')

for level in [2, 4]:
    ax.axhline(level, color='gray', linestyle='--', lw=1)

ax.set_ylim(0, 8)
ax.set_xlabel('$t$')
ax.set_ylabel('$x(t)$')
ax.set_title(r"Representative solutions for $x' = (4-x)(2-x)^3$")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

> **Tip**
>
> Notice how flat the curves are near $x=2$ compared with how sharply
> solutions bend away near $x=4$. This reflects $f'(2)=0$ versus
> $f'(4) = 8$: convergence toward a non-hyperbolic equilibrium (a
> repeated root) is much slower than the exponential rate predicted by
> Theorem 1.32 at a hyperbolic one.

------------------------------------------------------------------------

#### Exercise 3(c)

Use analytical or graphical methods to determine the equilibria of
$R' = \dfrac{3R}{1+R^2} - 1$.

**By Hand.**

Setting $f(R) = \dfrac{3R}{1+R^2} - 1 = 0$: $$
\frac{3R}{1+R^2} = 1 \quad \Longrightarrow \quad 3R = 1 + R^2 \quad \Longrightarrow \quad R^2 - 3R + 1 = 0.
$$

By the quadratic formula, $$
R = \frac{3 \pm \sqrt{9-4}}{2} = \frac{3 \pm \sqrt{5}}{2}.
$$

$$
\boxed{R_1 = \frac{3-\sqrt{5}}{2} \approx 0.382, \qquad R_2 = \frac{3+\sqrt{5}}{2} \approx 2.618}
$$

Graphically, these are exactly the two places where the curve $f(R)$
crosses the $R$-axis:

In [18]:
def f3c(R):
    return 3*R/(1+R**2) - 1

R = np.linspace(-1, 6, 500)
R1 = (3 - np.sqrt(5)) / 2
R2 = (3 + np.sqrt(5)) / 2

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(R, f3c(R), color='steelblue', lw=2)
ax.axhline(0, color='black', lw=1)
ax.plot(R1, 0, marker='o', ms=10, mfc='white', mec='crimson', mew=2, zorder=5)
ax.plot(R2, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax.annotate(f'$R_1 \\approx {R1:.3f}$', xy=(R1, 0), xytext=(R1-0.3, 0.6),
            fontsize=9, ha='right', arrowprops=dict(arrowstyle='->', lw=1))
ax.annotate(f'$R_2 \\approx {R2:.3f}$', xy=(R2, 0), xytext=(R2+0.3, 0.6),
            fontsize=9, ha='left', arrowprops=dict(arrowstyle='->', lw=1))
ax.set_xlabel('$R$')
ax.set_ylabel("$R' = f(R)$")
ax.set_title(r"Growth rate $f(R) = \dfrac{3R}{1+R^2} - 1$")
plt.tight_layout()
plt.show()

In [19]:
R = sym.symbols('R')
f = 3*R/(1+R**2) - 1
eqs = sym.solve(sym.Eq(f, 0), R)
print("Equilibria (exact):", eqs)
print("Equilibria (decimal):", [sym.N(e) for e in eqs])

Equilibria (exact): [3/2 - sqrt(5)/2, sqrt(5)/2 + 3/2]
Equilibria (decimal): [0.381966011250105, 2.61803398874989]

> **Tip**
>
> Although the exercise only asks for the equilibria, stability comes
> almost for free from the same plot: the curve crosses from $+$ to $-$
> at $R_2$ (stable) and from $-$ to $+$ at $R_1$ (unstable) — consistent
> with $f'(R_1) \approx 1.95 > 0$ and $f'(R_2) \approx -0.28 < 0$.

------------------------------------------------------------------------

#### Exercise 6 (Heat transfer)

Heat transfer by radiation from a body to its surroundings is modeled by
the Stefan–Boltzmann law $$
\frac{dT}{dt} = -k\left(T^4 - S^4\right),
$$ where $T = T(t)$ is the absolute temperature of the body and $S$ is
the (constant) temperature of the surroundings.

**(a) Phase line diagram and nature of the solution.**

The only equilibrium (for $T, S \geq 0$) is $T = S$, since $T^4 = S^4$
with $T,S\ge 0$ forces $T=S$. Writing $f(T) = -k(T^4-S^4)$:

- For $T > S$: $T^4 > S^4$, so $f(T) < 0$ (temperature decreasing — the
  body cools toward $S$).
- For $T < S$: $T^4 < S^4$, so $f(T) > 0$ (temperature increasing — the
  body warms toward $S$).

So $T = S$ is a **stable** equilibrium, and every solution approaches it
*monotonically* (no overshoot or oscillation) — exactly the physically
expected behavior of a body radiating heat until it reaches thermal
equilibrium with its surroundings.

In [20]:
k_demo = 2.0e-12
S_demo = 300

def fT(T, S):
    return -k_demo*(T**4 - S**4)

Tvals = np.linspace(0, 2*S_demo, 300)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5.5),
                                gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax1.plot(Tvals, fT(Tvals, S_demo), color='steelblue', lw=2)
ax1.axhline(0, color='black', lw=1)
ax1.plot(S_demo, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax1.set_ylabel("$T' = -k(T^4-S^4)$")
ax1.set_title("Growth rate for $T' = -k(T^4 - S^4)$")

bounds = [0, S_demo, 2*S_demo]
ax2.axhline(0, color='black', lw=1.5)
for a, b in zip(bounds[:-1], bounds[1:]):
    mid = (a + b) / 2
    sign = np.sign(fT(mid, S_demo))
    x0, x1 = (a + 0.05*S_demo, b - 0.05*S_demo) if sign > 0 else (b - 0.05*S_demo, a + 0.05*S_demo)
    ax2.annotate('', xy=(x1, 0), xytext=(x0, 0),
                 arrowprops=dict(arrowstyle='-|>', color='darkorange', lw=2))
ax2.plot(S_demo, 0, marker='o', ms=10, mfc='seagreen', mec='seagreen', zorder=5)
ax2.set_yticks([])
ax2.set_xlabel('$T$')
ax2.set_ylim(-1, 1)

plt.tight_layout()
plt.show()

**(b) Approximate solution neglecting $S$.**

With $T(0) = 2000\,\text{K}$ and $S = 300\,\text{K}$, since $S \ll T$ we
drop the $S^4$ term and solve $$
\frac{dT}{dt} = -kT^4, \qquad k = 2.0\times10^{-12}\ \text{K}^{-3}\text{sec}^{-1}.
$$

This is separable: $$
\frac{dT}{T^4} = -k\,dt \quad \Longrightarrow \quad -\frac{1}{3}T^{-3} = -kt + C \quad \Longrightarrow \quad T^{-3} = 3kt + T_0^{-3}.
$$

Solving for $T$ and substituting $T_0 = 2000$: $$
T(t) = \frac{T_0}{\left(1 + 3kT_0^3\,t\right)^{1/3}}.
$$

Since $3kT_0^3 = 3(2.0\times10^{-12})(2000)^3 = 0.048\ \text{sec}^{-1}$,

$$
\boxed{T(t) = \frac{2000}{(1+0.048\,t)^{1/3}} \ \text{K}, \qquad t \text{ in seconds}.}
$$

In [21]:
t = sym.symbols('t', positive=True)
T = sym.Function('T')
k_val = sym.Rational(2, 10**12)

eq = sym.Eq(sym.diff(T(t), t), -k_val*T(t)**4)
sol = sym.dsolve(eq, T(t), ics={T(0): 2000})
print("Solution:", sym.simplify(sol))
print("Coefficient 3*k*T0^3 =", sym.N(3*k_val*2000**3))

Solution: Eq(T(t), 10000/(6*t + 125)**(1/3))
Coefficient 3*k*T0^3 = 0.0480000000000000

> **Note**
>
> SymPy confirms
> $T(t)^{-3} = \dfrac{3t}{5\times10^{11}} + \dfrac{1}{8\times 10^9}$ (an
> equivalent way of writing the boxed formula), and the numerically
> computed coefficient matches $0.048$.

In [22]:
T0 = 2000
k = 2.0e-12
S = 300

t_vals = np.linspace(0, 3000, 400)
T_approx = T0 * (1 + 3*k*T0**3*t_vals)**(-1/3)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, T_approx, color='steelblue', lw=2, label=r'$T(t) = 2000(1+0.048t)^{-1/3}$')
ax.axhline(S, color='gray', linestyle='--', lw=1, label='$S = 300$ K')
ax.set_xlabel('$t$ (seconds)')
ax.set_ylabel('$T$ (K)')
ax.set_title('Approximate cooling curve, neglecting $S$')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

> **Tip**
>
> The approximate solution is only valid while $T \gg S$. By
> $t \approx 3000$ sec the body has cooled to around $380\,$K — close
> enough to $S = 300\,$K that neglecting $S^4$ in the original equation
> is no longer well justified; the full equation (including $S^4$) would
> need to be solved (numerically) to accurately track the final approach
> to equilibrium.

------------------------------------------------------------------------

### Section 1.5.2 — Bifurcations

For each model below (containing a parameter $h$): (a) find the
equilibria in terms of $h$ and determine their stability using Theorem
1.32, to the extent possible; (b) construct a bifurcation diagram
showing how the equilibria depend on $h$, labeling the branches as
stable or unstable.

#### Exercise 1(a): $x' = h - x^3$

**By Hand.**

Setting $f(x) = h - x^3 = 0$ gives the unique real equilibrium $$
x_{eq}(h) = h^{1/3}
$$ (the cube root function is one-to-one on $\mathbb{R}$, so there is
exactly one real equilibrium for every $h$; the other two roots of
$x^3=h$ are complex and not relevant here).

Differentiating, $f'(x) = -3x^2$, so $$
f'\!\left(x_{eq}\right) = -3h^{2/3} \leq 0 \quad \text{for every } h,
$$ with equality only at $h=0$. Thus $x_{eq}(h) = h^{1/3}$ is **stable
for every $h \neq 0$**. At $h=0$ (so $x_{eq}=0$), Theorem 1.32 is
inconclusive since $f'(0)=0$, but checking $f(x,0) = -x^3$ directly
shows $f>0$ for $x<0$ and $f<0$ for $x>0$ — trajectories are still drawn
toward $0$ from both sides, so $x=0$ remains stable (just
non-hyperbolically so).

> **Note**
>
> There is no actual bifurcation here: a single, always-stable branch of
> equilibria exists for every value of $h$, and no qualitative change in
> the equilibrium structure ever occurs. This example is a useful
> contrast to (b)–(d) below, where genuine bifurcations do occur.

In [23]:
h = np.linspace(-4, 4, 400)
x_eq = np.sign(h) * np.abs(h)**(1/3)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h, x_eq, color='seagreen', lw=2.5, label='stable')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$')
ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = h - x^3$")
ax.legend()
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 1(b): $x' = (x-1)(x-h)$

**By Hand.**

The equilibria are $x=1$ and $x=h$. Differentiating,
$f'(x) = (x-h)+(x-1) = 2x-(1+h)$, so $$
f'(1) = 1-h, \qquad f'(h) = h-1.
$$

- $x=1$ is **stable** when $h>1$ (so $f'(1)<0$) and **unstable** when
  $h<1$.
- $x=h$ is **stable** when $h<1$ (so $f'(h)<0$) and **unstable** when
  $h>1$.

At $h=1$, the two equilibria collide at $x=1$ and *exchange stability*
as $h$ increases through $1$ — this is the hallmark of a **transcritical
bifurcation**.

In [24]:
h_sym, x_sym = sym.symbols('h x')
f = (x_sym-1)*(x_sym-h_sym)
eqs = sym.solve(sym.Eq(f,0), x_sym)
fp = sym.diff(f, x_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(x_sym, e)))

Equilibria: [1, h]
  f'(1) = 1 - h
  f'(h) = h - 1

In [25]:
h = np.linspace(-3, 3, 400)
h_lo, h_hi = h[h <= 1], h[h >= 1]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h_lo, np.ones_like(h_lo), color='crimson', lw=2.5, linestyle='--')
ax.plot(h_hi, np.ones_like(h_hi), color='seagreen', lw=2.5)
ax.plot(h_lo, h_lo, color='seagreen', lw=2.5)
ax.plot(h_hi, h_hi, color='crimson', lw=2.5, linestyle='--')
ax.plot(1, 1, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = (x-1)(x-h)$")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 1(c): $y' = hy - y^2$

**By Hand.**

Factoring, $f(y) = y(h-y) = 0$ gives equilibria $y=0$ and $y=h$.
Differentiating, $f'(y) = h-2y$, so $$
f'(0) = h, \qquad f'(h) = -h.
$$

- $y=0$ is **stable** for $h<0$ and **unstable** for $h>0$.
- $y=h$ is **unstable** for $h<0$ and **stable** for $h>0$.

At $h=0$ the two equilibria merge at $y=0$ and swap stability — this is
the textbook normal form for a **transcritical bifurcation**.

In [26]:
h_sym, y_sym = sym.symbols('h y')
f = h_sym*y_sym - y_sym**2
eqs = sym.solve(sym.Eq(f,0), y_sym)
fp = sym.diff(f, y_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(y_sym, e)))

Equilibria: [0, h]
  f'(0) = h
  f'(h) = -h

In [27]:
h = np.linspace(-3, 3, 400)
h_lo, h_hi = h[h <= 0], h[h >= 0]

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(h_lo, np.zeros_like(h_lo), color='seagreen', lw=2.5)
ax.plot(h_hi, np.zeros_like(h_hi), color='crimson', lw=2.5, linestyle='--')
ax.plot(h_lo, h_lo, color='crimson', lw=2.5, linestyle='--')
ax.plot(h_hi, h_hi, color='seagreen', lw=2.5)
ax.plot(0, 0, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$y_{eq}$')
ax.set_title(r"Bifurcation diagram: $y' = hy - y^2$")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

#### Exercise 1(d): $x' = (1-x)(x^2-h)$

**By Hand.**

Setting $f(x) = (1-x)(x^2-h) = 0$ gives $x=1$ (for every $h$) together
with $x = \pm\sqrt{h}$ whenever $h \geq 0$ (no real square-root
equilibria when $h<0$). Differentiating, $$
f'(x) = -(x^2-h) + (1-x)(2x) = h + 2x - 3x^2,
$$ so $$
f'(1) = h - 1, \qquad f'(\sqrt{h}) = 2\sqrt{h}\left(1-\sqrt{h}\right), \qquad f'(-\sqrt{h}) = -2\sqrt{h}\left(1+\sqrt{h}\right).
$$

- $x=1$: **stable** for $h<1$, **unstable** for $h>1$.
- $x=-\sqrt{h}$ (exists for $h \geq 0$): since $\sqrt{h}\geq0$,
  $f'(-\sqrt h)\leq 0$ always $\Rightarrow$ **always stable**.
- $x=\sqrt{h}$ (exists for $h \geq 0$):
  $f'(\sqrt h) = 2\sqrt h(1-\sqrt h)$ is positive for $0<h<1$
  (**unstable**) and negative for $h>1$ (**stable**).

Two qualitative changes occur as $h$ varies:

- **At $h=0$:** for $h<0$ there is only the single equilibrium $x=1$; as
  $h$ increases through $0$, a pair of equilibria $x=\pm\sqrt h$ is born
  out of the degenerate double root at $x=0$ (where, at $h=0$,
  $f(x,0)=(1-x)x^2$ is semistable: attracting from the left, repelling
  from the right). This is a **saddle-node bifurcation**.
- **At $h=1$:** the branches $x=1$ and $x=\sqrt h$ meet (both equal $1$
  there) and exchange stability — a **transcritical bifurcation**,
  exactly as in parts (b) and (c).

In [28]:
h_sym, x_sym = sym.symbols('h x')
f = (1-x_sym)*(x_sym**2 - h_sym)
eqs = sym.solve(sym.Eq(f,0), x_sym)
fp = sym.diff(f, x_sym)
print("Equilibria:", eqs)
for e in eqs:
    print(f"  f'({e}) =", sym.simplify(fp.subs(x_sym, e)))

Equilibria: [1, -sqrt(h), sqrt(h)]
  f'(1) = h - 1
  f'(-sqrt(h)) = -2*sqrt(h) - 2*h
  f'(sqrt(h)) = 2*sqrt(h) - 2*h

In [29]:
h_pos = np.linspace(0, 3, 400)
h_all = np.linspace(-3, 3, 400)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

h_lo, h_hi = h_all[h_all <= 1], h_all[h_all >= 1]
ax.plot(h_lo, np.ones_like(h_lo), color='seagreen', lw=2.5)
ax.plot(h_hi, np.ones_like(h_hi), color='crimson', lw=2.5, linestyle='--')

h1, h2 = h_pos[h_pos <= 1], h_pos[h_pos >= 1]
ax.plot(h1, np.sqrt(h1), color='crimson', lw=2.5, linestyle='--')
ax.plot(h2, np.sqrt(h2), color='seagreen', lw=2.5)
ax.plot(h_pos, -np.sqrt(h_pos), color='seagreen', lw=2.5)

ax.plot(0, 0, marker='o', mfc='black', mec='black', ms=6)
ax.plot(1, 1, marker='o', mfc='black', mec='black', ms=6)

ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('$h$'); ax.set_ylabel('$x_{eq}$')
ax.set_title(r"Bifurcation diagram: $x' = (1-x)(x^2-h)$", pad=15)
ax.set_ylim(-1.9, 2.3)
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], color='seagreen', lw=2.5, label='stable'),
           Line2D([0], [0], color='crimson', lw=2.5, linestyle='--', label='unstable')]
ax.legend(handles=handles, loc='upper left')
ax.annotate('saddle-node\n(h=0)', xy=(0, 0), xytext=(-2.7, -1.6), fontsize=9,
            arrowprops=dict(arrowstyle='->'))
ax.annotate('transcritical\n(h=1)', xy=(1, 1), xytext=(1.7, 1.75), fontsize=9,
            arrowprops=dict(arrowstyle='->'))
plt.tight_layout()
plt.show()

------------------------------------------------------------------------

## References

> **Expand for Session Info**
>
> ``` python
> import sys # sys for system-specific parameters and functions
> print("Python version:", sys.version)
> print('\n'.join(f'{m.__name__}=={m.__version__}' for m in globals().values() if getattr(m, '__version__', None)))
> ```
>
>     Python version: 3.14.4 | packaged by conda-forge | (main, Apr  8 2026, 02:33:53) [Clang 20.1.8 ]
>     numpy==2.4.3
>     scipy==1.17.1
>     sympy==1.14.0
>     matplotlib==3.10.8

## Reuse

[![](http://mirrors.creativecommons.org/presskit/buttons/88x31/png/by-nc-sa.png?raw=1)](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode)

[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)